In [35]:
import scanpy as sc

adata = sc.read_h5ad(
    "/fs-computility-new/upzd_share/maoxinjie/AIVC/mxj/perturbench-main/data/Srivatsan2020_sciplex3_unseencell_hvgs.h5ad"
)




In [36]:
adata

AnnData object with n_obs × n_vars = 165299 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'batch', 'condition', 'gene_pt', 'drug_pt', 'env_pt', 'control', 'CRISPR', 'cell_cluster', 'dataset', 'split'
    var: 'ensembl_id', 'n_cells'
    uns: 'hvg_genes'
    layers: 'counts'

In [28]:
adata.obs["cell_cluster"].value_counts()

cell_cluster
K562    5752
Name: count, dtype: int64

In [24]:
adata.obs["perturbation"].value_counts()

perturbation
control    11855
KLF1        1960
BAK1        1457
CEBPE       1233
UBASH3B     1202
           ...  
FOXO4        215
NIT1         192
ZBTB10       162
HES7         126
PLK4         113
Name: count, Length: 106, dtype: int64

In [93]:
adata.obs["gene_pt"].value_counts()

gene_pt
         29143
GPX4     13348
LACZ      7567
OR2J2     7250
Name: count, dtype: int64

In [20]:
adata.obs['control'].value_counts()

control
False    57831
True     11855
Name: count, dtype: int64

In [14]:
adata.obs['split'].value_counts()

split
train    4045
val       854
test      853
Name: count, dtype: int64

In [84]:
import pandas as pd

pd.set_option('display.max_rows', None)
adata.obs['drug_pt'].value_counts()

drug_pt
          4430
nutlin    1096
dex        898
saha       540
bms        298
Name: count, dtype: int64

In [19]:
import pandas as pd
table = pd.crosstab(adata.obs['perturbation'], adata.obs['split'])
table

split,test,train,val
perturbation,,,
AHR,0,558,0
ARID1A,0,232,0
ARRDC3,0,495,0
ATL1,0,379,0
BAK1,0,1457,0
...,...,...,...
ZBTB10,0,162,0
ZBTB25,0,740,0
ZC3HAV1,0,516,0


In [119]:
adata.obs.loc[adata.obs['split'] == 'test', 'control'].value_counts()


control
False    5906
True     1185
Name: count, dtype: int64

In [118]:
adata.obs.loc[adata.obs['split'] == 'val', 'control'].value_counts()


control
False    5905
True     1186
Name: count, dtype: int64

In [117]:
adata.obs.loc[adata.obs['split'] == 'train', 'control'].value_counts()


control
False    46020
True      9484
Name: count, dtype: int64

In [6]:
import numpy as np
import torch
from pathlib import Path

npy_path = "/fs-computility-new/upzd_share/maoxinjie/AIVC/data/after_preprocess/perturbation_list/drug/drug.npy"
pt_path  = "/fs-computility-new/upzd_share/maoxinjie/AIVC/data/after_preprocess/perturbation_list/drug/drug.pt"

# 1. load npy
data_np = np.load(npy_path, allow_pickle=True)
print(type(data_np), data_np.shape)

# 2. 转成 torch
data_torch = torch.from_numpy(data_np)

# 3. 保存为 pt
torch.save(data_torch, pt_path)

print("Saved to:", pt_path)


<class 'numpy.ndarray'> (205, 2048)
Saved to: /fs-computility-new/upzd_share/maoxinjie/AIVC/data/after_preprocess/perturbation_list/drug/drug.pt


In [9]:
import torch

pt_path = "/fs-computility-new/upzd_share/maoxinjie/AIVC/mxj/perturbench-main/drug.pt"
data = torch.load(pt_path, map_location="cpu", weights_only=False)

print(type(data))        # torch.Tensor
print(data.shape)        # 重点
print(data.dtype)

# 看前几个元素
print(data[:10])


<class 'torch.Tensor'>
torch.Size([205, 2048])
torch.float32
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])


In [10]:
import anndata as ad

h5ad_path = "/fs-computility-new/upzd_share/shared/AIVC_data/processed_control/processed/LotfollahiTheis2023_processed.h5ad"
adata = ad.read_h5ad(h5ad_path)

# 看一下 drug_pt
adata.obs["drug_pt"].head()

# 原始 unique（可能含 A+B）
raw_unique = adata.obs["drug_pt"].dropna().unique()

# 拆分 A+B → A, B
drug_from_h5ad = set()
for x in raw_unique:
    for d in str(x).split("+"):
        drug_from_h5ad.add(d)

print("Unique drugs from h5ad:", len(drug_from_h5ad))

import torch

pt_path = "/fs-computility-new/upzd_share/maoxinjie/AIVC/mxj/perturbench-main/drug.pt"
drug_pt = torch.load(pt_path, map_location="cpu", weights_only=False)

print(type(drug_pt), drug_pt.shape)
print(drug_pt[:10])

drug_from_pt = set(map(str, drug_pt))

missing = drug_from_h5ad - drug_from_pt
extra   = drug_from_pt - drug_from_h5ad

print("❌ In h5ad but NOT in drug.pt:", len(missing))
print("➕ In drug.pt but NOT in h5ad:", len(extra))

list(missing)[:20]


Unique drugs from h5ad: 19
<class 'torch.Tensor'> torch.Size([205, 2048])
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
❌ In h5ad but NOT in drug.pt: 19
➕ In drug.pt but NOT in h5ad: 8


['',
 'pci-34051',
 'pci-34501',
 'sorafenib',
 'curcumin',
 'srt2104',
 'crizotinib',
 'srt3025',
 'tanespimycin',
 'dasatinib',
 'alvespimycin',
 'danusertib',
 'panobinostat',
 'givinostat',
 'dacinostat',
 'pirarubicin',
 'srt1720',
 'carmofur',
 'cediranib']